# Group Assignment - Part A: Practical Text Pre-Processing (Q5)

**Q5. Alternative Approach Implementation (Individual Component)**

For Q1, the group used NLTK's `word_tokenize` (whitespace/rule-based word tokenization) on `Data_1.txt`. In this section, I implement an **alternative tokenization approach — the BERT WordPiece Tokenizer** (from the Hugging Face `transformers` library) on the same corpus, and compare it against the group's NLTK-based approach to evaluate how each method handles subword splitting, hyphenated compounds, and out-of-vocabulary (OOV) words.

**Step 5.1: Install Dependencies**

The BERT tokenizer is not part of NLTK, so the Hugging Face `transformers` library is installed first. This library provides the pretrained `bert-base-uncased` WordPiece tokenizer used as the alternative tokenization technique for this question.

In [1]:
# Install the Hugging Face transformers library required for the BERT Tokenizer
!pip install transformers


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Step 5.2: Load the BERT Tokenizer**

The `BertTokenizer` class is imported so the pretrained `bert-base-uncased` model's WordPiece vocabulary can be used to tokenize the corpus.

In [2]:
from transformers import BertTokenizer

c:\Users\junyo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Step 5.3: Alternative Tokenization Implementation (BERT WordPiece Tokenizer) — 3 marks**

`Data_1.txt` (the same corpus used by the group in Q1) is loaded and tokenized using the pretrained `bert-base-uncased` WordPiece tokenizer. Unlike NLTK's `word_tokenize`, which splits text into whole words and punctuation, BERT's tokenizer breaks unfamiliar or longer words into smaller **subword units** (prefixed with `##` for continuation pieces) so that any input word can be represented using its fixed vocabulary. Both the string tokens and their corresponding numerical token IDs (including the special `[CLS]` and `[SEP]` tokens) are printed for inspection.

In [3]:
from transformers import BertTokenizer

with open('Data_1.txt', 'r', encoding='utf-8') as file:
    corpus_text = file.read()

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


bert_tokens = bert_tokenizer.tokenize(corpus_text)


token_ids = bert_tokenizer.encode(corpus_text)


total_strings = len(bert_tokens)
total_encodings = len(token_ids)

print(f"BERT Tokenization Complete!")
print(f"Total String Tokens Counted: {total_strings}")
print(f"Total Numerical Token IDs Counted: {total_encodings}")
print("-" * 50)


print("All WordPiece String Tokens:")
print(bert_tokens)

print("\nAll Corresponding Encodings (Includes [CLS] at the start and [SEP] at the end):")
print(token_ids)

c:\Users\junyo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\junyo\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


BERT Tokenization Complete!
Total String Tokens Counted: 97
Total Numerical Token IDs Counted: 99
--------------------------------------------------
All WordPiece String Tokens:
['classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'in', 'basic', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', '.', 'the', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants', '.', 'for', 'example', ',', 'in', 'multi', '##class', 'classification', ',', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', ';', 'in', 'open', '-', 'class', 'classification', ',', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', ';', 'and', 'in', 'sequence', 'classification', ',', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'classi

**Step 5.4: Compare and Contrast with the Group's NLTK Approach — 2 marks**

The BERT WordPiece token count is compared against the group's NLTK `word_tokenize` count on the same corpus, and three targeted mini-experiments illustrate the behavioural differences between the two tokenizers:
1. **Subword splitting** — whether a known word like *classification* is kept whole or broken into subword pieces.
2. **Hyphenated compound words** — whether *open-class* is kept as one token or split into `open`, `-`, `class`.
3. **Out-of-vocabulary (OOV) handling** — how each tokenizer handles a word that doesn't exist in a normal dictionary (`TechByteAccessories`).

**Why the alternative approach is better / worse / different — 5 marks**

- **Better:** BERT's subword (WordPiece) tokenization has no true OOV problem — any string can be represented as a sequence of known subword pieces (e.g. `tech`, `##by`, `##tea`, `##cc`, `##ess`, `##ories`), whereas NLTK's `word_tokenize` simply returns the unknown word as a single unsplit token with no internal structure. This makes BERT tokens far more useful as direct input to a neural language model, since the model can still extract partial meaning from familiar subword pieces even in rare or novel words.
- **Worse:** BERT's subword splitting produces less human-readable, less linguistically intuitive tokens (e.g. splitting *classification* into `classification` + `class` fragments in this run) and increases the total token count, which adds preprocessing complexity and makes the tokens harder to map back to whole dictionary words. NLTK's tokens map one-to-one with actual words, which is easier to interpret and to use for classic rule-based NLP tasks (e.g. POS tagging, parsing).
- **Just different:** NLTK's `word_tokenize` is a rule-based, linguistic-word tokenizer designed for classic NLP pipelines (tagging, parsing, information extraction), while BERT's tokenizer is a statistically-learned, fixed-vocabulary subword tokenizer purpose-built as the input layer for transformer language models. Neither is objectively "better" in general — the right choice depends on whether the downstream task is a traditional NLP pipeline (favouring NLTK) or a transformer-based deep learning model (favouring BERT).

In [4]:
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

# Generate standard NLTK tokens for comparison
tokens_nltk = word_tokenize(corpus_text)

print(f"Method 1: NLTK word_tokenize -> Total Tokens: {len(tokens_nltk)}")
print(f"Method 2: BERT Tokenizer     -> Total Tokens: {len(bert_tokens)}")

print("\n--- 1. Subword Splitting Experiment (Classification) ---")
# NLTK leaves words whole. BERT splits them using '##' to represent subword attachments.
nltk_sub = [t for t in tokens_nltk if 'Classification' in t or 'classification' in t][:1]
bert_sub = [t for t in bert_tokens if 'class' in t or '##fication' in t][:2]
print("NLTK Output :", nltk_sub)
print("BERT Output :", bert_sub)

print("\n--- 2. Compound Hyphenated Words Experiment (open-class) ---")
nltk_hyphen = [t for t in tokens_nltk if 'open-class' in t or t in ['open', '-', 'class']]
bert_hyphen = [t for t in bert_tokens if t in ['open', '-', 'class']]
print("NLTK Output :", nltk_hyphen)
print("BERT Output :", bert_hyphen)

print("\n--- 3. Out-Of-Vocabulary (OOV) Handling Simulation ---")
# If we test a rare or fake word like "TechByteAccessories"
fake_text = "TechByteAccessories"
print("NLTK Output on OOV word :", word_tokenize(fake_text))
print("BERT Output on OOV word :", bert_tokenizer.tokenize(fake_text))

Method 1: NLTK word_tokenize -> Total Tokens: 94
Method 2: BERT Tokenizer     -> Total Tokens: 97

--- 1. Subword Splitting Experiment (Classification) ---
NLTK Output : ['Classification']
BERT Output : ['classification', 'class']

--- 2. Compound Hyphenated Words Experiment (open-class) ---
NLTK Output : ['class', 'open-class']
BERT Output : ['class', 'open', '-', 'class']

--- 3. Out-Of-Vocabulary (OOV) Handling Simulation ---
NLTK Output on OOV word : ['TechByteAccessories']
BERT Output on OOV word : ['tech', '##by', '##tea', '##cc', '##ess', '##ories']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\junyo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
